In [ ]:
from scholar_rank.ingest.fetch_data import get_manifest, get_manifest_data, fetch_shard, extract_compact
from scholar_rank.utils import PROJECT_ROOT

# Fetching work entities from OpenAlex S3 storage
RAW_PATH: Path = PROJECT_ROOT/"data"/"openalex"
COMPACT_PATH: Path = PROJECT_ROOT/"data"/"compact"

manifest_path = RAW_PATH/"works"/"manifest.json"

In [ ]:
# Fixing keywords field prefix remnant.
# Rewrites each existing compact shard in place, stripping the leftover
# 'https://openalex.org/' prefix from keyword IDs (extract_compact() strips it
# correctly going forward; already-extracted shards predate that fix).

import os
import duckdb
from pathlib import Path
from scholar_rank.ingest.fetch_data import get_manifest_data

files = get_manifest_data(manifest_path)

db = duckdb.connect()

for i, file in enumerate(files):
    print(f"Fixing file {i+1} out of {len(files)}: {file.key}", flush=True)

    shard_path = COMPACT_PATH/file.key
    tmp_path = Path(str(shard_path) + ".tmp")

    rel = db.read_parquet(shard_path)

    rel = rel.select("""
        * REPLACE (
            list_transform(keywords, a -> {
                'id': replace(a.id, 'https://openalex.org/', ''),
                'display_name': a.display_name,
                'score': a.score
            }) AS keywords,
        ),
    """)

    # Write to a temp file first -- COPY reads shard_path lazily while writing,
    # so writing straight back to shard_path risks reading a partially
    # overwritten file (undefined behavior, possible corruption on the real corpus).
    db.sql(f"""
        COPY (SELECT * FROM rel)
        TO '{tmp_path}'
        (FORMAT parquet, COMPRESSION zstd)
    """)

    os.replace(tmp_path, shard_path)  # atomic on POSIX


In [ ]:
show_shard(COMPACT_PATH/sample_parquet)

In [ ]:
# Checking null-rate for all record in current local repo

import duckdb

columns = [
    "id",
    "doi",
    "title",
    "authorships",
    "abstract_inverted_index",
    "type",
    "language",
    "primary_location",
    "publication_year",
    "publication_date",
    "referenced_works",
    "referenced_works_count",
    "cited_by_count",
    "topics"
]
lst = "[" + ", ".join(f"'{c}'" for c in columns) + "]"

local_shard_list, remote_shard_list = list_local_and_remote_shards(manifest_path, RAW_PATH)

running_total = 0
running_null = dict.fromkeys(columns,0)

db = duckdb.connect()

print(f"{running_total} {running_null}")

for i in range(0,len(local_shard_list)):
    local = db.cursor()

    shard_path = local_shard_list[i]
    # print(f"Loading file {i+1} out of {len(local_shard_list)}: {shard_path}", flush=True)
    rel = local.read_parquet(str(RAW_PATH/shard_path)).aggregate(
        f'count(*) AS total, count(*) - count(COLUMNS({lst})) AS "\\0"'
    )
    row = rel.fetchone()
    running_total += row[0]
    for name, val in zip(rel.columns[1:], row[1:]):
        running_null[name] += val

with open("null_analysis.txt", "w", encoding="utf-8") as file:
    file.write(f"Total entries: {running_total}.\n")
    file.write(f"Columns labels:\n{columns}\n")
    file.write(f"Raw NULL count by column:\n{running_null}\n")
    file.write(f"Percent NULL by column\n{ {key: (float(value)/running_total) for key, value in running_null.items()} }\n")